# Yanjaro: bounded Yambda research

Real explicit-feedback proxy, not mood labels or Yandex track IDs. Dataset revision is pinned. No Render deployment. Start with a 128 MiB / 180 second range probe. Never fall back to downloading the entire 13.8 GB object.


In [ ]:
import os, sys, subprocess
from pathlib import Path
subprocess.run(['git','clone','--depth','1','--branch','experiment/remote-acoustic-yambda','https://github.com/zinverno/yanjaro.git','/content/yanjaro-research'],check=True)
os.chdir('/content/yanjaro-research/experiments/remote-acoustic')
os.environ['GITHUB_SHA']=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
os.environ['OPENBLAS_NUM_THREADS']='2'
subprocess.run([sys.executable,'-m','pip','install','.[train,serve,test]'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],check=True,timeout=120)


## Probe first

This reads one small row group through strict HTTP ranges. Arrow batch size does not bound network bytes. Inspect projected bytes and timing before choosing any larger scan. Failure stops this cell; no model has been trained.


In [ ]:
subprocess.run([sys.executable,'-m','remote_acoustic.cli','probe'],check=True,timeout=240)


## Prepare real feedback

Only two small likes/dislikes files. Anonymous events stay in temporary work/ and must not be published. Context precedes both targets, and holdout users are disjoint from training.


In [ ]:
subprocess.run([sys.executable,'-m','remote_acoustic.cli','prepare','--work','work'],check=True,timeout=480)


## Bounded extraction

At most 64 batches, 512 MiB payload, 600 seconds. This is partial coverage and may be insufficient to train. A full scan is a separate reviewed run; do not remove limits to make this notebook pass.


In [ ]:
subprocess.run([sys.executable,'-m','remote_acoustic.cli','extract','--work','work','--max-batches','64','--max-read-mib','512','--max-seconds','600'],check=True,timeout=640)


## Train once if coverage is sufficient

Fixed hyperparameters. No holdout tuning. Lower learned accuracy must be reported honestly. Failed extraction or insufficient coverage prevents export.


In [ ]:
subprocess.run([sys.executable,'-m','remote_acoustic.cli','train','--work','work','--output','artifacts/model.json'],check=True,timeout=300)
import json
report=json.loads(Path('artifacts/metrics.json').read_text())
print(json.dumps(report,indent=2))


## Download model and aggregate metrics only

The data card limits the dataset to research. These weights do not accept the player's 27 handcrafted features. Catalog mapping, commercial-use review and deployment remain separate stages.


In [ ]:
from google.colab import files
model=json.loads(Path('artifacts/model.json').read_text())
assert model['source']=='yandex/yambda'
assert model['provenance']['dataset_revision']=='dd6f3a19eef5866e346c3270e098baa641a44948'
files.download('artifacts/model.json')
files.download('artifacts/metrics.json')
